# SPY Next-Day Direction Project

##### Predict next-day SPY direction over the past 2 years with Logistic Regression, Single Decision Tree and Random Forest. Retrain monthly during the test period using an expanding window. Compare predictive performance and over/underfitting on train and test.

1. **Build a tree by hand** using Gini impurity and one manual split example.
2. **Train one in sklearn** using `DecisionTreeClassifier`, with readable tree rules and plotted tree.
3. **Tame overfitting** using `max_depth`, `min_samples_leaf`, and **cost-complexity pruning path** with `ccp_alpha`.
4. **Train a Random Forest** using many trees, `max_features='sqrt'`, `oob_score=True`, and `feature_importances_`.
5. **Predict next-day SPY direction over the past 2 years**, retraining monthly with an expanding window.
6. **Compare predictive performance and over/underfitting on train and test**.

- Features are known at time $t$. Target is next-day direction from $t$ to $t+1$.
- Walk-forward training removes the final pre-test row each month to avoid using a label whose return occurs inside the next test month.

> Research notebook for educational purposes only — not investment advice. See [Disclaimer](#disclaimer) at the bottom.


In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import os

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

plt.style.use("ggplot")
plt.rcParams["axes.grid"] = False

def safe_roc_auc(y_true, y_score):
    """
    ROC-AUC is undefined when a train/test month has only one class.
    Return np.nan instead of crashing the notebook.
    """
    try:
        if len(np.unique(y_true)) < 2:
            return np.nan
        return roc_auc_score(y_true, y_score)
    except Exception:
        return np.nan

## USER-CHOSEN SETTINGS (HYPERPARAMETERS)

In [2]:
START_DATE = "2020-01-01"
TEST_YEARS = 2
MIN_TRAIN_ROWS = 50
RANDOM_STATE = 42

# Logistic Regression
LOGREG_MAX_ITER = 1000
LOGREG_C = 1.0

# Single Decision Tree
TREE_MAX_DEPTH = 3
TREE_MIN_SAMPLES_SPLIT = 20
TREE_MIN_SAMPLES_LEAF = 10

# Random Forest
RF_N_ESTIMATORS = 500
RF_MAX_DEPTH = 3
RF_MIN_SAMPLES_SPLIT = 20
RF_MIN_SAMPLES_LEAF = 10
RF_MAX_FEATURES = "sqrt"

# Hyperparameter sensitivity grids for tree models
TREE_DEPTH_GRID = [1, 2, 3, 4, 5, 6, 8, 10]
TREE_LEAF_GRID = [1, 2, 5, 10, 20, 30, 50]
CCP_ALPHA_GRID_SIZE = 20

# Output folder
# Fixed for Mac/Jupyter/Terminal:
# The original code used Path("spy_project_output"), which depends on the
# current terminal working directory. If Python is launched from a read-only
# location, mkdir() fails with: OSError: [Errno 30] Read-only file system.
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    # Jupyter notebook does not define __file__
    BASE_DIR = Path.cwd()

if not os.access(BASE_DIR, os.W_OK):
    # Fallback for macOS when current directory is not writable
    BASE_DIR = Path.home() / "Desktop"

OUTDIR = BASE_DIR / "spy_project_output"
OUTDIR.mkdir(parents=True, exist_ok=True)

print(f"Output folder ready: {OUTDIR.resolve()}")

Output folder ready: /Users/shivanshmittal/Github/ml-market-direction-models/spy_project_output


## 1. Download SPY data

In [3]:
spy = yf.download(
    "SPY",
    start=START_DATE,
    auto_adjust=True,
    progress=False,
    multi_level_index=False
)

if spy is None or spy.empty:
    raise ValueError("No data downloaded from Yahoo Finance.")

if isinstance(spy.columns, pd.MultiIndex):
    spy.columns = spy.columns.get_level_values(0)

df = spy[["Close"]].copy()

## 2. Feature engineering

In [4]:
df["r"] = df["Close"].pct_change()

# =========================================================
# Feature timing rule:
# Every feature must be known at time t.
# The target and tradable return are both for t+1.
# =========================================================
df["lag1"] = df["r"].shift(1)
df["lag2"] = df["r"].shift(2)
df["lag3"] = df["r"].shift(3)
df["lag5"] = df["r"].shift(5)
df["ma5"] = df["r"].rolling(5).mean().shift(1)

# Correct next-day design:
# At date t, features are known up to time t, and the label is whether SPY return at t+1 is positive.
# The strategy PnL must also use this same next_day_return, not same-day r.
df["next_day_return"] = df["r"].shift(-1)
df["target"] = np.where(
    df["next_day_return"].isna(),
    np.nan,
    (df["next_day_return"] > 0).astype(int)
)

# Keep raw engineered data before dropping NaNs.
# This is useful for walk-forward retraining, but rows with unknown next_day_return are removed before modeling.
full_df = df.copy()

# Final cleaned modeling data
features = ["lag1", "lag2", "lag3", "lag5", "ma5"]
df = df.dropna(subset=features + ["next_day_return", "target"]).copy()
df["target"] = df["target"].fillna(-1).astype(int)
df = df[df["target"] != -1].copy()

print("Data after preprocessing:")
print(df[["Close", "r", "next_day_return", "target"] + features].head())
print("\nTotal usable rows:", len(df))
print("\nTiming audit:")
print("Feature date t  →  predicts target t+1  →  earns next_day_return t+1")


Data after preprocessing:
                 Close         r  next_day_return  target      lag1      lag2  \
Date                                                                            
2020-01-10  296.890961 -0.002878         0.006877       1  0.006781  0.005329   
2020-01-13  298.932770  0.006877        -0.001524       0 -0.002878  0.006781   
2020-01-14  298.477051 -0.001524         0.002260       1  0.006877 -0.002878   
2020-01-15  299.151550  0.002260         0.008318       1 -0.001524  0.006877   
2020-01-16  301.639954  0.008318         0.003113       1  0.002260 -0.001524   

                lag3      lag5       ma5  
Date                                      
2020-01-10 -0.002811 -0.007572  0.001108  
2020-01-13  0.005329  0.003815  0.002047  
2020-01-14  0.006781 -0.002811  0.002660  
2020-01-15 -0.002878  0.005329  0.002917  
2020-01-16  0.006877  0.006781  0.002303  

Total usable rows: 1626

Timing audit:
Feature date t  →  predicts target t+1  →  earns next_day_return 

## Leakage audit conclusion

- Features use only information known before the prediction.
- `target` is based on `next_day_return = r.shift(-1)`.
- Strategy returns also use `next_day_return`, so prediction timing and PnL timing now match.
- In walk-forward monthly retraining, the final training row before each test month is removed so the training label does not depend on the first test-month return.


## 3. Global train/test split

In [5]:
last_date = df.index.max()
test_start = pd.Timestamp(last_date - pd.DateOffset(years=TEST_YEARS)).normalize()
test_start = df.index[df.index >= test_start][0]  # snap to first actual trading day

print("\nLast date in sample:", last_date.date())
print("Test window starts:", test_start.date())
print("NOTE: test_start is pinned to the nearest trading day for reproducibility.")


Last date in sample: 2026-07-01
Test window starts: 2024-07-01
NOTE: test_start is pinned to the nearest trading day for reproducibility.


## 4. Define models

In [6]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=LOGREG_MAX_ITER,
            C=LOGREG_C,
            random_state=RANDOM_STATE
        ))
    ]),
    "Single Decision Tree": DecisionTreeClassifier(
        max_depth=TREE_MAX_DEPTH,
        min_samples_split=TREE_MIN_SAMPLES_SPLIT,
        min_samples_leaf=TREE_MIN_SAMPLES_LEAF,
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        max_depth=RF_MAX_DEPTH,
        min_samples_split=RF_MIN_SAMPLES_SPLIT,
        min_samples_leaf=RF_MIN_SAMPLES_LEAF,
        max_features=RF_MAX_FEATURES,
        oob_score=True,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
}

## 5. Initial leakage-guarded train period

In [7]:
initial_train_raw = full_df.loc[full_df.index < test_start].copy()
initial_train_raw = initial_train_raw.iloc[:-1].copy()
initial_train_df = initial_train_raw.dropna(subset=features + ["next_day_return", "target"]).copy()
initial_train_df["target"] = initial_train_df["target"].fillna(-1).astype(int)
initial_train_df = initial_train_df[initial_train_df["target"] != -1].copy()

initial_test_raw = full_df.loc[full_df.index >= test_start].copy()
initial_test_df = initial_test_raw.dropna(subset=features + ["next_day_return", "target"]).copy()
initial_test_df["target"] = initial_test_df["target"].fillna(-1).astype(int)
initial_test_df = initial_test_df[initial_test_df["target"] != -1].copy()

X_train_init = initial_train_df[features]
y_train_init = initial_train_df["target"]

X_test_init = initial_test_df[features]
y_test_init = initial_test_df["target"]


## 6. Build a tiny tree by hand using Gini impurity

Before reaching for `sklearn`, it's worth seeing a split chosen by hand: pick splits with Gini, entropy, or SSE and explain why each is best.

We use one simple candidate split: `lag1 > 0`.

The goal is not to manually rebuild the entire sklearn algorithm. The goal is to show how a classification tree chooses a split: it compares impurity before and after the split and prefers the split that reduces impurity the most.


In [8]:
def gini_impurity(y):
    """Manual Gini impurity for binary classification."""
    y = pd.Series(y).dropna()
    if len(y) == 0:
        return np.nan
    p = y.value_counts(normalize=True)
    return 1 - np.sum(p ** 2)

# Candidate manual split: lag1 > 0
manual_rule = X_train_init["lag1"] > 0
left_y = y_train_init[~manual_rule]   # lag1 <= 0
right_y = y_train_init[manual_rule]   # lag1 > 0

parent_gini = gini_impurity(y_train_init)
left_gini = gini_impurity(left_y)
right_gini = gini_impurity(right_y)

left_weight = len(left_y) / len(y_train_init)
right_weight = len(right_y) / len(y_train_init)
weighted_child_gini = left_weight * left_gini + right_weight * right_gini
gini_reduction = parent_gini - weighted_child_gini

hand_tree_pred = manual_rule.astype(int)
hand_tree_acc = accuracy_score(y_train_init, hand_tree_pred)

manual_gini_table = pd.DataFrame({
    "Node": ["Parent", "Left: lag1 <= 0", "Right: lag1 > 0", "Weighted children"],
    "Rows": [len(y_train_init), len(left_y), len(right_y), len(y_train_init)],
    "Gini": [parent_gini, left_gini, right_gini, weighted_child_gini],
    "Weight": [1.0, left_weight, right_weight, 1.0]
})
manual_gini_table["Gini"] = manual_gini_table["Gini"].round(4)
manual_gini_table["Weight"] = manual_gini_table["Weight"].round(4)
manual_gini_table.to_csv(OUTDIR / "manual_gini_hand_tree_example.csv", index=False)

print("Manual Gini split example")
print("Rule: if lag1 > 0, predict Up; otherwise predict Down/Flat")
print(manual_gini_table)
print("\nGini reduction from this split:", round(gini_reduction, 6))
print("Hand-built one-split rule training accuracy:", round(hand_tree_acc, 4))

if gini_reduction > 0:
    print("Interpretation: this split reduces impurity, so it has some classification value.")
else:
    print("Interpretation: this split does not reduce impurity much; sklearn will search other features/thresholds.")


Manual Gini split example
Rule: if lag1 > 0, predict Up; otherwise predict Down/Flat
                Node  Rows    Gini  Weight
0             Parent  1123  0.4966  1.0000
1    Left: lag1 <= 0   515  0.4965  0.4586
2    Right: lag1 > 0   608  0.4966  0.5414
3  Weighted children  1123  0.4966  1.0000

Gini reduction from this split: 0.0
Hand-built one-split rule training accuracy: 0.5031
Interpretation: this split reduces impurity, so it has some classification value.


## 7. Fit initial models

In [9]:
initial_fit_rows = []
fitted_initial_models = {}

for name, model in models.items():
    model.fit(X_train_init, y_train_init)
    fitted_initial_models[name] = model

    pred_train = model.predict(X_train_init)
    pred_test = model.predict(X_test_init)

    if hasattr(model, "predict_proba"):
        proba_train = model.predict_proba(X_train_init)[:, 1]
        proba_test = model.predict_proba(X_test_init)[:, 1]
    else:
        proba_train = pred_train.astype(float)
        proba_test = pred_test.astype(float)

    initial_fit_rows.append({
        "Model": name,
        "Train Accuracy": accuracy_score(y_train_init, pred_train),
        "Test Accuracy": accuracy_score(y_test_init, pred_test),
        "Train Precision": precision_score(y_train_init, pred_train, zero_division=0),
        "Test Precision": precision_score(y_test_init, pred_test, zero_division=0),
        "Train Recall": recall_score(y_train_init, pred_train, zero_division=0),
        "Test Recall": recall_score(y_test_init, pred_test, zero_division=0),
        "Train F1": f1_score(y_train_init, pred_train, zero_division=0),
        "Test F1": f1_score(y_test_init, pred_test, zero_division=0),
        "Train ROC_AUC": safe_roc_auc(y_train_init, proba_train),
        "Test ROC_AUC": safe_roc_auc(y_test_init, proba_test)
    })

initial_fit_results = pd.DataFrame(initial_fit_rows)
initial_fit_results.to_csv(OUTDIR / "initial_fit_results.csv", index=False)
print("\nInitial fit results:")
print(initial_fit_results.round(4))


if hasattr(fitted_initial_models["Random Forest"], "oob_score_"):
    print("\nRandom Forest OOB score:", round(fitted_initial_models["Random Forest"].oob_score_, 4))



Initial fit results:
                  Model  Train Accuracy  Test Accuracy  Train Precision  \
0   Logistic Regression          0.5494         0.5697           0.5469   
1  Single Decision Tree          0.5744         0.5339           0.5714   
2         Random Forest          0.5993         0.5518           0.5779   

   Test Precision  Train Recall  Test Recall  Train F1  Test F1  \
0          0.5706        0.9786       0.9895    0.7017   0.7238   
1          0.5610        0.8553       0.8357    0.6851   0.6713   
2          0.5639        0.9638       0.9406    0.7226   0.7051   

   Train ROC_AUC  Test ROC_AUC  
0         0.5153        0.5315  
1         0.5669        0.4724  
2         0.7038        0.4675  

Random Forest OOB score: 0.5387


## 8. Show a sklearn decision tree

In [10]:
simple_tree = fitted_initial_models["Single Decision Tree"]
tree_rules = export_text(simple_tree, feature_names=features)

with open(OUTDIR / "08_single_tree_rules.txt", "w", encoding="utf-8") as f:
    f.write(tree_rules)

print("\nSklearn decision tree structure:")
print(tree_rules)


Sklearn decision tree structure:
|--- lag3 <= 0.02
|   |--- lag3 <= 0.01
|   |   |--- ma5 <= 0.00
|   |   |   |--- class: 1
|   |   |--- ma5 >  0.00
|   |   |   |--- class: 0
|   |--- lag3 >  0.01
|   |   |--- lag3 <= 0.01
|   |   |   |--- class: 1
|   |   |--- lag3 >  0.01
|   |   |   |--- class: 1
|--- lag3 >  0.02
|   |--- lag2 <= -0.01
|   |   |--- class: 0
|   |--- lag2 >  -0.01
|   |   |--- lag1 <= 0.01
|   |   |   |--- class: 0
|   |   |--- lag1 >  0.01
|   |   |   |--- class: 0



## 9. Monthly walk-forward test

In [11]:
# Monthly retraining starts from the month that contains test_start.
# Important fix: for the first test month, do NOT include dates before the exact 2-year test_start.
# Example: if test_start is 2024-05-18, the May test slice starts at 2024-05-18, not 2024-05-01.
test_month_starts = pd.date_range(
    start=test_start.to_period("M").to_timestamp(),
    end=last_date,
    freq="MS"
)

rows = []
daily_test_preds = []
confusion_rows = []

for month_start in test_month_starts:
    month_end = month_start + pd.offsets.MonthEnd(1)

    # Expanding-window monthly retraining:
    # Train uses only observations strictly before the month being tested.
    # For first test month, test_start_exact removes the pre-test days inside that calendar month.
    test_start_exact = max(month_start, test_start)

    train_raw = full_df.loc[full_df.index < month_start].copy()
    test_raw = full_df.loc[(full_df.index >= test_start_exact) & (full_df.index <= month_end)].copy()

    if len(train_raw) < 60 or len(test_raw) == 0:
        continue

    train_raw = train_raw.iloc[:-1].copy()

    train_df = train_raw.dropna(subset=features + ["next_day_return", "target"]).copy()
    test_df = test_raw.dropna(subset=features + ["next_day_return", "target"]).copy()
    train_df["target"] = train_df["target"].astype(int)
    test_df["target"] = test_df["target"].astype(int)

    if len(train_df) < MIN_TRAIN_ROWS or len(test_df) == 0:
        continue

    X_train = train_df[features]
    y_train = train_df["target"]

    X_test = test_df[features]
    y_test = test_df["target"]

    baseline_pred_train = np.ones(len(y_train), dtype=int)
    baseline_pred_test = np.ones(len(y_test), dtype=int)
    baseline_proba_train = np.ones(len(y_train), dtype=float)
    baseline_proba_test = np.ones(len(y_test), dtype=float)

    rows.append({
        "Month": month_start.strftime("%Y-%m"),
        "Model": "Always Predict Up Baseline",
        "Train Accuracy": accuracy_score(y_train, baseline_pred_train),
        "Test Accuracy": accuracy_score(y_test, baseline_pred_test),
        "Train Precision": precision_score(y_train, baseline_pred_train, zero_division=0),
        "Test Precision": precision_score(y_test, baseline_pred_test, zero_division=0),
        "Train Recall": recall_score(y_train, baseline_pred_train, zero_division=0),
        "Test Recall": recall_score(y_test, baseline_pred_test, zero_division=0),
        "Train F1": f1_score(y_train, baseline_pred_train, zero_division=0),
        "Test F1": f1_score(y_test, baseline_pred_test, zero_division=0),
        "Train ROC_AUC": safe_roc_auc(y_train, baseline_proba_train),
        "Test ROC_AUC": safe_roc_auc(y_test, baseline_proba_test),
        "Train Size": len(train_df),
        "Test Size": len(test_df)
    })

    cm_base = confusion_matrix(y_test, baseline_pred_test, labels=[0, 1])
    confusion_rows.append({
        "Model": "Always Predict Up Baseline",
        "Month": month_start.strftime("%Y-%m"),
        "TN": cm_base[0, 0],
        "FP": cm_base[0, 1],
        "FN": cm_base[1, 0],
        "TP": cm_base[1, 1]
    })

    from sklearn.base import clone
    for name, model in models.items():
        fitted_model = clone(model)
        fitted_model.fit(X_train, y_train)
        model = fitted_model  # use fitted_model below instead of model to avoid data leakage in next iterations

        pred_train = fitted_model.predict(X_train)
        pred_test = fitted_model.predict(X_test)

        if hasattr(fitted_model, "predict_proba"):
            proba_train = fitted_model.predict_proba(X_train)[:, 1]
            proba_test = fitted_model.predict_proba(X_test)[:, 1]
        else:
            proba_train = pred_train.astype(float)
            proba_test = pred_test.astype(float)

        rows.append({
            "Month": month_start.strftime("%Y-%m"),
            "Model": name,
            "Train Accuracy": accuracy_score(y_train, pred_train),
            "Test Accuracy": accuracy_score(y_test, pred_test),
            "Train Precision": precision_score(y_train, pred_train, zero_division=0),
            "Test Precision": precision_score(y_test, pred_test, zero_division=0),
            "Train Recall": recall_score(y_train, pred_train, zero_division=0),
            "Test Recall": recall_score(y_test, pred_test, zero_division=0),
            "Train F1": f1_score(y_train, pred_train, zero_division=0),
            "Test F1": f1_score(y_test, pred_test, zero_division=0),
            "Train ROC_AUC": safe_roc_auc(y_train, proba_train),
            "Test ROC_AUC": safe_roc_auc(y_test, proba_test),
            "Train Size": len(train_df),
            "Test Size": len(test_df)
        })

        daily_test_preds.append(pd.DataFrame({
            "Date": test_df.index,
            "Model": name,
            "Actual Return": test_df["next_day_return"].values,
            "Actual Target": y_test.values,
            "Prediction": pred_test,
            "Probability": proba_test
        }))

        cm = confusion_matrix(y_test, pred_test, labels=[0, 1])
        confusion_rows.append({
            "Model": name,
            "Month": month_start.strftime("%Y-%m"),
            "TN": cm[0, 0],
            "FP": cm[0, 1],
            "FN": cm[1, 0],
            "TP": cm[1, 1]
        })

results = pd.DataFrame(rows)
confusion_df = pd.DataFrame(confusion_rows)

if results.empty:
    raise ValueError("No walk-forward results were produced.")

daily_test_predictions = pd.concat(daily_test_preds, ignore_index=True)

results.to_csv(OUTDIR / "monthly_walkforward_results.csv", index=False)
confusion_df.to_csv(OUTDIR / "monthly_confusion_matrices.csv", index=False)
daily_test_predictions.to_csv(OUTDIR / "daily_test_predictions.csv", index=False)


## 10. Summary comparison

In [12]:
summary = (
    results.groupby("Model")[[
        "Train Accuracy", "Test Accuracy",
        "Train Precision", "Test Precision",
        "Train Recall", "Test Recall",
        "Train F1", "Test F1",
        "Train ROC_AUC", "Test ROC_AUC"
    ]]
    .mean()
    .round(4)
)

summary["Accuracy Gap"] = (summary["Train Accuracy"] - summary["Test Accuracy"]).round(4)
summary["F1 Gap"] = (summary["Train F1"] - summary["Test F1"]).round(4)

def diagnose(row):
    if row["Accuracy Gap"] > 0.10 or row["F1 Gap"] > 0.10:
        return "Possible overfitting"
    elif row["Train Accuracy"] < 0.55 and row["Test Accuracy"] < 0.55:
        return "Possible underfitting"
    else:
        return "Reasonable generalization"

summary["Diagnosis"] = summary.apply(diagnose, axis=1)
summary.to_csv(OUTDIR / "summary_results.csv")

print("\nSummary results:")
print(summary)


Summary results:
                            Train Accuracy  Test Accuracy  Train Precision  \
Model                                                                        
Always Predict Up Baseline          0.5487         0.5483           0.5487   
Logistic Regression                 0.5588         0.5520           0.5557   
Random Forest                       0.5834         0.5464           0.5704   
Single Decision Tree                0.5683         0.5351           0.5618   

                            Test Precision  Train Recall  Test Recall  \
Model                                                                   
Always Predict Up Baseline          0.5483        1.0000       0.9600   
Logistic Regression                 0.5524        0.9776       0.9508   
Random Forest                       0.5519        0.9761       0.9353   
Single Decision Tree                0.5451        0.9722       0.9180   

                            Train F1  Test F1  Train ROC_AUC  Test ROC_AUC

## 11. Baseline comparison

In [13]:
baseline_name = "Always Predict Up Baseline"
baseline_test_acc = summary.loc[baseline_name, "Test Accuracy"]

ml_models = [m for m in summary.index if m != baseline_name]
baseline_compare = summary.loc[ml_models, ["Test Accuracy"]].copy()
baseline_compare["Always-Up Baseline Accuracy"] = baseline_test_acc
baseline_compare["Accuracy Minus Baseline"] = (
    baseline_compare["Test Accuracy"] - baseline_compare["Always-Up Baseline Accuracy"]
).round(4)

baseline_compare.to_csv(OUTDIR / "baseline_comparison.csv")
print("\nBaseline comparison:")
print(baseline_compare)



Baseline comparison:
                      Test Accuracy  Always-Up Baseline Accuracy  \
Model                                                              
Logistic Regression          0.5520                       0.5483   
Random Forest                0.5464                       0.5483   
Single Decision Tree         0.5351                       0.5483   

                      Accuracy Minus Baseline  
Model                                          
Logistic Regression                    0.0037  
Random Forest                         -0.0019  
Single Decision Tree                  -0.0132  


## 12. Full test confusion matrices

In [14]:
full_test_confusions = []

for model_name in ["Always Predict Up Baseline"] + list(models.keys()):
    if model_name == "Always Predict Up Baseline":
        y_true = initial_test_df["target"].values
        y_pred = np.ones(len(y_true), dtype=int)
    else:
        temp = daily_test_predictions[daily_test_predictions["Model"] == model_name].copy()
        y_true = temp["Actual Target"].values
        y_pred = temp["Prediction"].values

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    full_test_confusions.append({
        "Model": model_name,
        "TN": cm[0, 0],
        "FP": cm[0, 1],
        "FN": cm[1, 0],
        "TP": cm[1, 1]
    })

full_test_confusion_df = pd.DataFrame(full_test_confusions)
full_test_confusion_df.to_csv(OUTDIR / "full_test_confusion_matrix.csv", index=False)
print("\nFull test confusion matrices:")
print(full_test_confusion_df)



Full test confusion matrices:
                        Model  TN   FP  FN   TP
0  Always Predict Up Baseline   0  216   0  286
1         Logistic Regression   5  211   3  283
2        Single Decision Tree   6  210  13  273
3               Random Forest   7  209   8  278


## 13. Train/test equity curves

In [15]:
train_growth = pd.DataFrame(index=initial_train_df.index)
train_growth["SPY Buy & Hold"] = (1 + initial_train_df["next_day_return"]).cumprod()

from sklearn.base import clone
for name, model in models.items():
    fresh_model = clone(model)
    fresh_model.fit(X_train_init, y_train_init)
    pred_train_full = fresh_model.predict(X_train_init)

    strat_ret = np.where(pred_train_full == 1, initial_train_df["next_day_return"].values, 0.0)
    train_growth[name] = (1 + pd.Series(strat_ret, index=initial_train_df.index)).cumprod()

test_growth = pd.DataFrame(index=initial_test_df.index)
test_growth["SPY Buy & Hold"] = (1 + initial_test_df["next_day_return"]).cumprod()

for model_name in daily_test_predictions["Model"].unique():
    model_df = daily_test_predictions[daily_test_predictions["Model"] == model_name].copy()
    model_df = model_df.sort_values("Date")
    model_df["Strategy Return"] = np.where(
        model_df["Prediction"] == 1,
        model_df["Actual Return"],
        0.0
    )
    model_df = model_df.set_index("Date")
    aligned = model_df.reindex(initial_test_df.index)
    aligned["Strategy Return"] = aligned["Strategy Return"].fillna(0.0)
    test_growth[model_name] = (1 + aligned["Strategy Return"]).cumprod()

test_growth["Always Predict Up Baseline"] = (1 + initial_test_df["next_day_return"]).cumprod()

train_growth.to_csv(OUTDIR / "train_growth_of_1_dollar.csv")
test_growth.to_csv(OUTDIR / "test_growth_of_1_dollar.csv")


## 14. Hyperparameter sensitivity: decision tree
Compare train/test accuracy as max_depth changes
and as min_samples_leaf changes.

In [16]:
depth_rows = []
for depth in TREE_DEPTH_GRID:
    model = DecisionTreeClassifier(
        max_depth=depth,
        min_samples_split=TREE_MIN_SAMPLES_SPLIT,
        min_samples_leaf=TREE_MIN_SAMPLES_LEAF,
        random_state=RANDOM_STATE
    )
    model.fit(X_train_init, y_train_init)

    pred_train = model.predict(X_train_init)
    pred_test = model.predict(X_test_init)

    depth_rows.append({
        "max_depth": depth,
        "Train Accuracy": accuracy_score(y_train_init, pred_train),
        "Test Accuracy": accuracy_score(y_test_init, pred_test),
        "Train F1": f1_score(y_train_init, pred_train, zero_division=0),
        "Test F1": f1_score(y_test_init, pred_test, zero_division=0)
    })

depth_df = pd.DataFrame(depth_rows)
depth_df.to_csv(OUTDIR / "tree_depth_sensitivity.csv", index=False)

leaf_rows = []
for leaf in TREE_LEAF_GRID:
    model = DecisionTreeClassifier(
        max_depth=TREE_MAX_DEPTH,
        min_samples_split=TREE_MIN_SAMPLES_SPLIT,
        min_samples_leaf=leaf,
        random_state=RANDOM_STATE
    )
    model.fit(X_train_init, y_train_init)

    pred_train = model.predict(X_train_init)
    pred_test = model.predict(X_test_init)

    leaf_rows.append({
        "min_samples_leaf": leaf,
        "Train Accuracy": accuracy_score(y_train_init, pred_train),
        "Test Accuracy": accuracy_score(y_test_init, pred_test),
        "Train F1": f1_score(y_train_init, pred_train, zero_division=0),
        "Test F1": f1_score(y_test_init, pred_test, zero_division=0)
    })

leaf_df = pd.DataFrame(leaf_rows)
leaf_df.to_csv(OUTDIR / "tree_leaf_sensitivity.csv", index=False)

## 15. Cost-complexity pruning path with `ccp_alpha`

Taming overfitting: `max_depth`, `min_samples_leaf`, `ccp_alpha`, and the cost-complexity path.

A deep decision tree can memorize noisy daily market patterns. Cost-complexity pruning adds a penalty for tree complexity. Higher `ccp_alpha` produces a smaller tree.

Interpretation:
- Very small `ccp_alpha`: tree is flexible and can overfit.
- Very large `ccp_alpha`: tree becomes too simple and can underfit.
- Best region: smaller train/test gap and stronger test performance.

**Note:** this pruning-path section is a diagnostic/teaching section. It illustrates how `ccp_alpha` changes overfitting and underfitting. The test accuracy shown here is used for explanation and comparison, not as a production hyperparameter-selection procedure.


In [17]:
# Full deep tree first: this is the overfitting benchmark.
deep_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
deep_tree.fit(X_train_init, y_train_init)

path = deep_tree.cost_complexity_pruning_path(X_train_init, y_train_init)
ccp_alphas = path.ccp_alphas
impurities = path.impurities

# Remove the last alpha because it usually prunes the tree to only the root.
ccp_alphas = ccp_alphas[:-1]
impurities = impurities[:-1]

if len(ccp_alphas) > CCP_ALPHA_GRID_SIZE:
    alpha_index = np.linspace(0, len(ccp_alphas) - 1, CCP_ALPHA_GRID_SIZE).astype(int)
    ccp_alphas_to_test = np.unique(ccp_alphas[alpha_index])
else:
    ccp_alphas_to_test = ccp_alphas

pruning_rows = []
for alpha in ccp_alphas_to_test:
    pruned_tree = DecisionTreeClassifier(
        random_state=RANDOM_STATE,
        ccp_alpha=float(alpha)
    )
    pruned_tree.fit(X_train_init, y_train_init)

    pred_train = pruned_tree.predict(X_train_init)
    pred_test = pruned_tree.predict(X_test_init)

    pruning_rows.append({
        "ccp_alpha": float(alpha),
        "Train Accuracy": accuracy_score(y_train_init, pred_train),
        "Test Accuracy": accuracy_score(y_test_init, pred_test),
        "Train F1": f1_score(y_train_init, pred_train, zero_division=0),
        "Test F1": f1_score(y_test_init, pred_test, zero_division=0),
        "Tree Depth": pruned_tree.get_depth(),
        "Leaves": pruned_tree.get_n_leaves()
    })

pruning_df = pd.DataFrame(pruning_rows)
pruning_df["Accuracy Gap"] = pruning_df["Train Accuracy"] - pruning_df["Test Accuracy"]
pruning_df["F1 Gap"] = pruning_df["Train F1"] - pruning_df["Test F1"]

# Diagnostic choice for classroom explanation only:
# show the alpha with high test accuracy and a smaller train-test gap.
# This is NOT presented as final production tuning; it is used to demonstrate overfitting control.
best_pruning_row = pruning_df.sort_values(
    by=["Test Accuracy", "Accuracy Gap"],
    ascending=[False, True]
).iloc[0]

pruning_df.to_csv(OUTDIR / "cost_complexity_pruning_path_results.csv", index=False)

print("Cost-complexity pruning path results:")
print(pruning_df.round(5))
print("\nBest ccp_alpha by test accuracy with gap tie-breaker:")
print(best_pruning_row.round(5))

print("\nOverfitting interpretation:")
print("- If train accuracy is much higher than test accuracy, the tree is likely memorizing noise.")
print("- Pruning reduces complexity by shrinking depth/leaves, usually reducing variance.")


Cost-complexity pruning path results:
    ccp_alpha  Train Accuracy  Test Accuracy  Train F1  Test F1  Tree Depth  \
0     0.00000         1.00000        0.52390   1.00000  0.57699          25   
1     0.00074         0.99377        0.52789   0.99425  0.58053          25   
2     0.00080         0.98308        0.52988   0.98446  0.58451          25   
3     0.00082         0.97596        0.52390   0.97781  0.57549          25   
4     0.00089         0.96616        0.52390   0.96865  0.57699          25   
5     0.00107         0.95637        0.51992   0.95967  0.57496          25   
6     0.00117         0.94301        0.52191   0.94719  0.57597          25   
7     0.00119         0.93411        0.52390   0.93884  0.57848          25   
8     0.00127         0.91897        0.52390   0.92510  0.58579          25   
9     0.00132         0.91095        0.53187   0.91803  0.59552          25   
10    0.00139         0.89938        0.53386   0.90622  0.59233          22   
11    0.00142 

## 16. Random Forest feature importance and OOB check

`n_estimators=500`, `max_features='sqrt'`, `oob_score=True`, compared via `feature_importances_`.


In [18]:
rf_model = fitted_initial_models["Random Forest"]

rf_importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

rf_importance_df.to_csv(OUTDIR / "random_forest_feature_importance_table.csv", index=False)

print("Random Forest parameters used:")
print("n_estimators =", RF_N_ESTIMATORS)
print("max_features =", RF_MAX_FEATURES)
print("oob_score = True")
if hasattr(rf_model, "oob_score_"):
    print("OOB score =", round(rf_model.oob_score_, 4))

print("\nRandom Forest feature importances:")
print(rf_importance_df.round(5))


Random Forest parameters used:
n_estimators = 500
max_features = sqrt
oob_score = True
OOB score = 0.5387

Random Forest feature importances:
  Feature  Importance
2    lag3     0.23813
4     ma5     0.21241
0    lag1     0.20394
3    lag5     0.17616
1    lag2     0.16937


## 17. Helper function for heatmaps

In [19]:
def annotated_heatmap(data, title, filename, cmap="YlGnBu", fmt=".3f"):
    fig, ax = plt.subplots(figsize=(12, max(5, 0.5 * len(data.index) + 1.5)))
    im = ax.imshow(data.values, aspect="auto", cmap=cmap)

    ax.set_xticks(np.arange(len(data.columns)))
    ax.set_xticklabels(data.columns, rotation=45, ha="right", fontsize=11)
    ax.set_yticks(np.arange(len(data.index)))
    ax.set_yticklabels(data.index, fontsize=11)

    ax.set_title(title, fontsize=15, pad=12)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            val = data.iloc[i, j]
            ax.text(
                j, i, format(val, fmt),
                ha="center",
                va="center",
                color="black",
                fontsize=10,
                fontweight="bold"
            )

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.tick_params(labelsize=10)

    fig.tight_layout()
    fig.savefig(OUTDIR / filename, dpi=170, bbox_inches="tight")
    plt.close(fig)

## 18. Plots

In [20]:
# Time split on SPY price
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(full_df.index, full_df["Close"], color="tab:blue", linewidth=1.6, label="SPY Close")
ax.axvspan(test_start, full_df.index.max(), color="tab:blue", alpha=0.12, label="Out-of-sample test period")
ax.axvline(test_start, color="tab:blue", linestyle="--", linewidth=1.6, label=f"Test start: {test_start.date()}")
ax.set_title("Time Split on SPY Price: Train vs Out-of-Sample Test", fontsize=16)
ax.set_xlabel("Date")
ax.set_ylabel("SPY Close")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUTDIR / "01_time_split_spy_price.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Train vs test return distribution
fig, ax = plt.subplots(figsize=(12, 6))
ax.hist(initial_train_df["next_day_return"], bins=40, alpha=0.60, label="Train", color="tab:blue", edgecolor="none")
ax.hist(initial_test_df["next_day_return"], bins=40, alpha=0.60, label="Test", color="tab:orange", edgecolor="none")
ax.axvline(0, color="black", linestyle="--", linewidth=1.4, label="Zero return")
ax.set_title("Next-Day Return Distribution: Train vs Test", fontsize=16)
ax.set_xlabel("Next-day return")
ax.set_ylabel("Frequency")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUTDIR / "02_train_test_return_distribution.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Monthly test accuracy
monthly_test_acc = results.pivot(index="Month", columns="Model", values="Test Accuracy")
fig, ax = plt.subplots(figsize=(12, 6))
for col in monthly_test_acc.columns:
    ax.plot(monthly_test_acc.index, monthly_test_acc[col], marker="o", linewidth=1.7, label=col)
ax.set_title("Monthly Walk-Forward Test Accuracy", fontsize=15)
ax.set_xlabel("Month")
ax.set_ylabel("Accuracy")
ax.tick_params(axis="x", rotation=60)
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUTDIR / "03_monthly_test_accuracy.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Average train vs test accuracy
plot_df = summary.reset_index()
x = np.arange(len(plot_df))
width = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width/2, plot_df["Train Accuracy"], width, label="Train Accuracy", color="tab:blue")
ax.bar(x + width/2, plot_df["Test Accuracy"], width, label="Test Accuracy", color="tab:orange")
ax.set_title("Average Train vs Test Accuracy", fontsize=15)
ax.set_xticks(x)
ax.set_xticklabels(plot_df["Model"], rotation=15)
ax.set_ylabel("Accuracy")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(OUTDIR / "04_train_vs_test_accuracy.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Train growth of $1
fig, ax = plt.subplots(figsize=(14, 6))
for col in train_growth.columns:
    ax.plot(train_growth.index, train_growth[col], label=col, linewidth=1.9)
ax.set_title("Train Period: Growth of $1", fontsize=16)
ax.set_xlabel("Date")
ax.set_ylabel("Portfolio Value ($)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUTDIR / "05_train_growth_of_1_dollar.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Test growth of $1
fig, ax = plt.subplots(figsize=(14, 6))
for col in test_growth.columns:
    ax.plot(test_growth.index, test_growth[col], label=col, linewidth=1.9)
ax.set_title("Test Period: Growth of $1", fontsize=16)
ax.set_xlabel("Date")
ax.set_ylabel("Portfolio Value ($)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUTDIR / "06_test_growth_of_1_dollar.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Single decision tree structure
tree_model = fitted_initial_models["Single Decision Tree"]
fig, ax = plt.subplots(figsize=(18, 10))
plot_tree(
    tree_model,
    feature_names=features,
    class_names=["Down/Flat", "Up"],
    filled=True,
    rounded=True,
    impurity=True,
    proportion=True,
    fontsize=10,
    ax=ax
)
ax.set_title("Single Decision Tree Structure", fontsize=16, pad=14)
fig.tight_layout()
fig.savefig(OUTDIR / "07_single_decision_tree_structure.png", dpi=190, bbox_inches="tight")
plt.close(fig)

# Random forest feature importance
rf_model = fitted_initial_models["Random Forest"]
rf_importance = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
rf_importance.plot(kind="barh", ax=ax, color="tab:green")
ax.set_title("Random Forest Feature Importance", fontsize=15)
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")
ax.grid(True, axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(OUTDIR / "09_random_forest_feature_importance.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Logistic probability histogram
log_model = fitted_initial_models["Logistic Regression"]
log_test_prob = log_model.predict_proba(X_test_init)[:, 1]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(log_test_prob, bins=25, color="tab:orange", edgecolor="black", alpha=0.80)
ax.axvline(0.5, color="black", linestyle="--", linewidth=1.5, label="0.5 cutoff")
ax.set_title("Logistic Regression: Test Predicted Probability of Up", fontsize=15)
ax.set_xlabel("Predicted probability of Up")
ax.set_ylabel("Frequency")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(OUTDIR / "10_logistic_test_probability_histogram.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Confusion matrix heatmaps
for model_name in full_test_confusion_df["Model"]:
    row = full_test_confusion_df[full_test_confusion_df["Model"] == model_name].iloc[0]
    cm_data = pd.DataFrame(
        [[row["TN"], row["FP"]], [row["FN"], row["TP"]]],
        index=["Actual 0 Down/Flat", "Actual 1 Up"],
        columns=["Pred 0 Down/Flat", "Pred 1 Up"]
    )
    safe_name = model_name.lower().replace(" ", "_").replace("/", "_")
    annotated_heatmap(
        cm_data,
        f"Full Test Confusion Matrix - {model_name}",
        f"11_confusion_matrix_{safe_name}.png",
        cmap="Blues",
        fmt=".0f"
    )

# Hyperparameter sensitivity: max_depth
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depth_df["max_depth"], depth_df["Train Accuracy"], marker="o", linewidth=1.8, label="Train Accuracy")
ax.plot(depth_df["max_depth"], depth_df["Test Accuracy"], marker="o", linewidth=1.8, label="Test Accuracy")
ax.set_title("Decision Tree Performance vs Max Depth", fontsize=15)
ax.set_xlabel("max_depth")
ax.set_ylabel("Accuracy")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUTDIR / "12_tree_performance_vs_max_depth.png", dpi=180, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(depth_df["max_depth"], depth_df["Train F1"], marker="o", linewidth=1.8, label="Train F1")
ax.plot(depth_df["max_depth"], depth_df["Test F1"], marker="o", linewidth=1.8, label="Test F1")
ax.set_title("Decision Tree F1 vs Max Depth", fontsize=15)
ax.set_xlabel("max_depth")
ax.set_ylabel("F1")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUTDIR / "13_tree_f1_vs_max_depth.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Hyperparameter sensitivity: min_samples_leaf
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(leaf_df["min_samples_leaf"], leaf_df["Train Accuracy"], marker="o", linewidth=1.8, label="Train Accuracy")
ax.plot(leaf_df["min_samples_leaf"], leaf_df["Test Accuracy"], marker="o", linewidth=1.8, label="Test Accuracy")
ax.set_title("Decision Tree Performance vs Min Samples Leaf", fontsize=15)
ax.set_xlabel("min_samples_leaf")
ax.set_ylabel("Accuracy")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUTDIR / "14_tree_performance_vs_min_samples_leaf.png", dpi=180, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(leaf_df["min_samples_leaf"], leaf_df["Train F1"], marker="o", linewidth=1.8, label="Train F1")
ax.plot(leaf_df["min_samples_leaf"], leaf_df["Test F1"], marker="o", linewidth=1.8, label="Test F1")
ax.set_title("Decision Tree F1 vs Min Samples Leaf", fontsize=15)
ax.set_xlabel("min_samples_leaf")
ax.set_ylabel("F1")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUTDIR / "15_tree_f1_vs_min_samples_leaf.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Heatmaps
heatmap_models = [
    "Logistic Regression",
    "Single Decision Tree",
    "Random Forest",
    "Always Predict Up Baseline"
]

monthly_acc_hm = results[results["Model"].isin(heatmap_models)].pivot(
    index="Month", columns="Model", values="Test Accuracy"
)
annotated_heatmap(
    monthly_acc_hm,
    "Monthly Test Accuracy Heatmap",
    "20_monthly_test_accuracy_heatmap.png",
    cmap="YlGnBu",
    fmt=".3f"
)

monthly_f1_hm = results[results["Model"].isin(heatmap_models)].pivot(
    index="Month", columns="Model", values="Test F1"
)
annotated_heatmap(
    monthly_f1_hm,
    "Monthly Test F1 Heatmap",
    "21_monthly_test_f1_heatmap.png",
    cmap="YlOrRd",
    fmt=".3f"
)

gap_df = results.copy()
gap_df["Accuracy Gap"] = gap_df["Train Accuracy"] - gap_df["Test Accuracy"]
monthly_gap_hm = gap_df[gap_df["Model"].isin(heatmap_models)].pivot(
    index="Month", columns="Model", values="Accuracy Gap"
)
annotated_heatmap(
    monthly_gap_hm,
    "Monthly Accuracy Gap Heatmap (Train - Test)",
    "22_monthly_accuracy_gap_heatmap.png",
    cmap="coolwarm",
    fmt=".3f"
)



# Cost-complexity pruning path plots
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(pruning_df["ccp_alpha"], pruning_df["Train Accuracy"], marker="o", linewidth=1.8, label="Train Accuracy")
ax.plot(pruning_df["ccp_alpha"], pruning_df["Test Accuracy"], marker="o", linewidth=1.8, label="Test Accuracy")
ax.set_title("Cost-Complexity Pruning: Accuracy vs ccp_alpha", fontsize=15)
ax.set_xlabel("ccp_alpha")
ax.set_ylabel("Accuracy")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUTDIR / "16_cost_complexity_accuracy_vs_alpha.png", dpi=180, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(pruning_df["ccp_alpha"], pruning_df["Tree Depth"], marker="o", linewidth=1.8, label="Tree Depth")
ax.plot(pruning_df["ccp_alpha"], pruning_df["Leaves"], marker="o", linewidth=1.8, label="Leaves")
ax.set_title("Cost-Complexity Pruning: Tree Size vs ccp_alpha", fontsize=15)
ax.set_xlabel("ccp_alpha")
ax.set_ylabel("Tree complexity")
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(OUTDIR / "17_cost_complexity_tree_size_vs_alpha.png", dpi=180, bbox_inches="tight")
plt.close(fig)

# Manual Gini table heatmap
manual_gini_hm = manual_gini_table.set_index("Node")[["Gini", "Weight"]]
annotated_heatmap(
    manual_gini_hm,
    "Manual Gini Split Example",
    "18_manual_gini_split_example_heatmap.png",
    cmap="YlGnBu",
    fmt=".4f"
)

print(f"\nFiles saved in: {OUTDIR.resolve()}")
print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print("-", p.name)



Files saved in: /Users/shivanshmittal/Github/ml-market-direction-models/spy_project_output
Saved files:
- 01_time_split_spy_price.png
- 02_train_test_return_distribution.png
- 03_monthly_test_accuracy.png
- 04_train_vs_test_accuracy.png
- 05_train_growth_of_1_dollar.png
- 06_test_growth_of_1_dollar.png
- 07_single_decision_tree_structure.png
- 08_single_tree_rules.txt
- 09_random_forest_feature_importance.png
- 10_logistic_test_probability_histogram.png
- 11_confusion_matrix_always_predict_up_baseline.png
- 11_confusion_matrix_logistic_regression.png
- 11_confusion_matrix_random_forest.png
- 11_confusion_matrix_single_decision_tree.png
- 12_tree_performance_vs_max_depth.png
- 13_tree_f1_vs_max_depth.png
- 14_tree_performance_vs_min_samples_leaf.png
- 15_tree_f1_vs_min_samples_leaf.png
- 16_cost_complexity_accuracy_vs_alpha.png
- 17_cost_complexity_tree_size_vs_alpha.png
- 18_manual_gini_split_example_heatmap.png
- 20_monthly_test_accuracy_heatmap.png
- 21_monthly_test_f1_heatmap.png
-

---

<a id="disclaimer"></a>
## Disclaimer

This notebook is shared for research and educational purposes only and does not constitute investment advice. Next-day direction prediction on a single, heavily-arbitraged index (SPY) is an intentionally hard, low-signal problem — the point here is the methodology (leakage-safe walk-forward evaluation, overfitting diagnostics) rather than a claim of tradeable edge.
